### **BUILDING WHOLE GPT-2 MOE**

#### **IMPORTS NECESARIOS**

In [1]:
import sys
import torch
import torch.nn as nn
from pathlib import Path


#### **1. INTEGRATING MOE IN TRANFORMER BLOCK**

##### **1.1 ATTENTION**

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in:int, d_out:int, context_length:int, attr:dict):

        # Initialiaze the parent
        super().__init__() 

        # Get att specific params
        num_heads:int = attr.get('num_heads', 6)
        dropout:float = attr.get('dropout', 0.1)
        qkv_bias:bool = attr.get('qkv_bias', False)

        # Get basic parameters
        self.d_out = d_out
        self.num_heads = num_heads
        # Force sum(dim(heads)) =  dout
        self.head_dim = d_out // num_heads 

        # Define the QKV matrices -> dk = dv = dout
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal = 1)   
        )

    def forward(self, x):
        # Get the shape (B, N, d_in)
        b, num_tokens, d_in = x.shape
        # Define the KQV matrices (B, num_tokens, d_out)
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We modify shapes to consider all the heads
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention 
        attn_scores = queries @ keys.transpose(2, 3)  

        # Original mask truncated to the number of tokens and converted to boolean
        # Addapted to the real amount of tokens in case input is not padded to obtain fixed N (context_length)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores -> upper diagonal MLM
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # Get attention weights
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where d_out -> num_heads * head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

##### **1.2 NORMALIZATION**

In [11]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim:int):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x:torch.Tensor)->torch.Tensor:
        """
            Scales data per instance and makes
            it follow a ~N(0, 1)
        """
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

##### **1.3 MOE BLOCK**

In [4]:
src_path = str(Path("/home/carlo/cuarto/GPT2_MOE/src").resolve())

# Asegurar que src tiene prioridad
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

from gpt2_moe.model.moe import MOE

##### **1.4 TRANSFORMER BLOCK**

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parent
        super().__init__()
        # Common params
        self.d_in = attr["emb_dim"]
        self.d_out = attr["emb_dim"]
        self.cl = attr["context_length"]
        # Block specific params
        self.att_related = attr["attention_related"]
        self.moe_related = attr["moe_related"]
        
        # Define the attention module -> ok
        self.att = MultiHeadAttention(
           self.d_in,
           self.d_out,
           self.cl,
           self.att_related)

        # Define the ff layer
        self.ff = MOE(
                    self.d_in, 
                    self.d_out, 
                    self.moe_related) 

        # Define normalizations (within and previous)
        self.norm1 = LayerNorm(self.d_in)
        self.norm2 = LayerNorm(self.d_in)
        
        # Define the dropout
        self.drop_shortcut = nn.Dropout(attr['dropout'])

    def forward(self, x:torch.Tensor)->torch.Tensor:
        # Info message
        print('\nENTERING A TRANSFORMER BLOCK')
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        # (B, N, de) ->(B, N, de)
        x = self.att(x)  
        print(f'(TFBLOCK) Aspect of attention output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut  

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        # Go through ff Layer
        x = self.ff(x)
        print(f'(TFBLOCK) Aspect of FF output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut 

        print(f'(TFBLOCK) Aspect of block output ({x.shape})')
 

        return x

In [6]:
# Define config for the block
block_config = {
    "context_length": 1024, 
    "emb_dim": 768, 
    "dropout":0.1,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":False
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    }
}

In [25]:
# Define mock tensor -> currently embedded
mock = torch.randn(size = (10, 100, 768))

# Get the block
tf_block = TransformerBlock(block_config)

# Test the function
result = tf_block(mock)




ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 3])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 3])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([353, 768])
(expert) Output of the second linear:torch.Size([353, 768])
(expert) Aspect of the expert weights: torch.Size([353, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.Size([332, 768])
(expert) Output of the second linear:torch.Size([332, 768])
(expe

#### **2. INTEGRATING MOE IN GPT BLOCK**


##### **2.1 GPT BLOCK**

In [30]:
class GPTMOE(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parents
        super().__init__()
        
        # Define embedding layer
        self.tok_emb = nn.Embedding(attr["vocab_size"], attr["emb_dim"])
        self.pos_emb = nn.Embedding(attr["context_length"], attr["emb_dim"])
        self.drop_emb = nn.Dropout(attr["dropout"])

        # Build the transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(attr) for _ in range(attr["n_layers"])])


        # Output layer norm
        self.final_norm = LayerNorm(attr["emb_dim"])
        # Language head
        self.out_head = nn.Linear(
            attr["emb_dim"], attr["vocab_size"], bias=False
        )

    def forward(self, in_idx:torch.Tensor)->torch.Tensor:

        # Get input dims (B, N)
        batch_size, seq_len = in_idx.shape
        # Embed the inputs (B, N, de)
        # 1. Get contextual embeddings
        tok_embeds = self.tok_emb(in_idx)
        
        # 2. Get positional embeddings
        print(f'(GPT)CREATING EMBEEDINGS')
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        print(f'(GPT)Aspect of the embeddings:{x.shape}')

        # 3. Go trought transformer blocks (B, N, de) -> (B, N, de)
        x = self.trf_blocks(x)
        print(f'(GPT)Output of the trf blocks:{x.shape}')
        x = self.final_norm(x)

        # Get logits (B, N, de) ->(B, N, |V|)
        logits = self.out_head(x)
        print(f'(GPT)Aspect of the logits space:{x.shape}')

        return logits

##### **2.2 PARAM DEFINITION**

In [31]:
# Define config for the block
block_config = {
    "vocab_size": 50257, 
    "context_length": 1024,
    "n_layers": 16,  
    "emb_dim": 768, 
    "dropout":0.1,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":False
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    }
}

##### **2.3 TESTING**

In [32]:
# Create the gpt2 object
gpt2_moe = GPTMOE(block_config)

# Define the mock tensor -> (B, N, dtype = int)
mock_tensor = torch.randint(low = 10, high = 100, size = (10, 100))
gpt2_moe(mock_tensor)


(GPT)CREATING EMBEEDINGS
(GPT)Aspect of the embeddings:torch.Size([10, 100, 768])

ENTERING A TRANSFORMER BLOCK
(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 3])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 3])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([387, 768])
(expert) Output of the second linear:torch.Size([387, 768])
(expert) Aspect of the expert weights: torch.Size([387, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.

tensor([[[ 0.1738, -0.0101, -0.4489,  ..., -0.4159, -0.5228,  0.4444],
         [-0.4300,  0.9108,  0.5463,  ...,  0.3157,  0.0830,  1.2109],
         [-0.4661, -0.1206, -0.4901,  ..., -0.5371,  0.1251,  0.5306],
         ...,
         [-0.0513,  0.6896,  0.0165,  ...,  0.1455,  1.0126,  0.0529],
         [ 0.0254, -0.0989,  0.0280,  ...,  0.2468,  0.2781,  0.2258],
         [-0.5712,  0.4311,  0.7278,  ..., -0.2744,  0.1209, -0.1562]],

        [[ 0.5134, -0.4642, -0.5055,  ..., -0.2418, -0.6185, -0.2352],
         [-0.6065, -0.1439, -0.7081,  ..., -0.2279,  0.1032, -0.2604],
         [-0.1529, -0.2274, -1.1154,  ...,  0.3768, -0.4220,  0.0918],
         ...,
         [ 0.4759,  0.9963, -0.2378,  ...,  1.0035,  0.1291,  0.0212],
         [ 0.0575,  0.4590, -1.0369,  ...,  0.4694,  0.1936,  0.1237],
         [-0.4316,  0.2320,  0.8680,  ...,  0.0067,  0.1968, -0.7201]],

        [[ 0.6360, -1.4944, -0.7697,  ..., -0.3461,  0.3974,  0.2772],
         [-0.6934, -0.2645, -0.3382,  ...,  0